# Чтение данных

In [1]:
from typing import Generator

import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05

In [2]:
df = pd.read_csv("ab_experiment_data.csv")
df.sample(5)

,user_id,exp_group,order_id,order_sum,profit_from_order
265606,74901,test,265607,1973.544888,394.708978
53111,35500,control,53112,1870.434124,374.086825
42928,28678,control,42929,1959.487866,391.897573
209881,39007,test,209882,1873.545666,374.709133
29377,19668,control,29378,1864.749626,372.949925


# Пункт (a)
**Удалось ли вырастить кол-во заказов между группами?**

Чтобы проверить данную гипотезу, мы можем отследить метрику среднего количества заказов по пользователю в каждой из групп. Для тестирования гипотезы будем использовать `t-test`.

In [3]:
control_counts = df[df["exp_group"] == "control"].groupby("user_id").size().values
test_counts = df[df["exp_group"] == "test"].groupby("user_id").size().values

point_estimate = test_counts.mean() - control_counts.mean()
print(f"Point estimate of the difference: {point_estimate:.3f}")

test_result = stats.ttest_ind(test_counts, control_counts, equal_var=False)
pvalue = test_result.pvalue

print(f"P-value: {pvalue:.3f}")
if pvalue < ALPHA:
    print("Reject the null hypothesis")
else:
    print("Reject the alternative hypothesis")

Point estimate of the difference: 0.104
P-value: 0.000
Reject the null hypothesis


# Пункт (b)
**Как мы повлияли на общую сумму, которую пользователи оплатили за заказы?**

Чтобы проверить данную гипотезу, будем проверять метрику общей выручки с пользователя в каждой из групп. Для тестирования гипотезы будем использовать `t-test`.

In [4]:
control_income = (
    df[df["exp_group"] == "control"].groupby("user_id")["order_sum"].sum().values
)
test_income = df[df["exp_group"] == "test"].groupby("user_id")["order_sum"].sum().values

point_estimate = test_income.mean() - control_income.mean()
print(f"Point estimate of the difference in income: {point_estimate:.3f}")

test_result = stats.ttest_ind(test_income, control_income, equal_var=False)
pvalue = test_result.pvalue

print(f"P-value: {pvalue:.3f}")
if pvalue < ALPHA:
    print("Reject the null hypothesis")
else:
    print("Reject the alternative hypothesis")

Point estimate of the difference in income: -102.348
P-value: 0.000
Reject the null hypothesis


# Пункт (c)
**Как наше изменение повлияло на средний чек заказов на сервисе?**

Чтобы проверить данную гипотезу, будем проверять метрику среднего чека за заказ в каждой из групп. Для тестирования гипотезы будем использовать `t-test`.


In [5]:
control_bill = (
    df[df["exp_group"] == "control"].groupby("order_id")["order_sum"].mean().values
)
test_bill = df[df["exp_group"] == "test"].groupby("order_id")["order_sum"].mean().values

point_estimate = test_bill.mean() - control_bill.mean()
print(f"Point estimate of the difference in income: {point_estimate:.3f}")

test_result = stats.ttest_ind(test_bill, control_bill, equal_var=False)
pvalue = test_result.pvalue

print(f"P-value: {pvalue:.3f}")
if pvalue < ALPHA:
    print("Reject the null hypothesis")
else:
    print("Reject the alternative hypothesis")

Point estimate of the difference in income: -99.995
P-value: 0.000
Reject the null hypothesis


# Пункт (d)

**Как наше изменение повлияло на суммарную прибыль сервиса? Сколько рублей прибыли мы теряем или получаем за один инкрементальный заказ?**

Нам надо посмотреть на изменение общей прибыли сервиса. Чтобы это сделать, сравним метрику общей прибыли между группами. Так как у нас нет способа посчитать дисперсию выручки, будем использовать `bootstrap` с квантильным доверительным интервалом для проверки гипотезы.

In [6]:
def bootstrap(data: np.ndarray, size: int = 10000) -> Generator[np.ndarray, None, None]:
    n = len(data)
    for _ in range(size):
        yield np.random.choice(data, size=n, replace=True)


control_revenue = df[df["exp_group"] == "control"]["profit_from_order"].values
test_revenue = df[df["exp_group"] == "test"]["profit_from_order"].values

control_revenue_bootstrap_means = np.array([np.mean(sample) for sample in bootstrap(control_revenue)])
test_revenue_bootstrap_means = np.array([np.mean(sample) for sample in bootstrap(test_revenue)])

statistics = test_revenue_bootstrap_means - control_revenue_bootstrap_means
point_estimate = test_revenue.mean() - control_revenue.mean()

print(f"Point estimate of the difference in revenue: {point_estimate:.3f}")

ci = np.quantile(statistics, [ALPHA / 2, 1 - ALPHA / 2])
print(
    f"{1 - ALPHA:.2f} confidence interval of the difference in revenue: [{ci[0]:.3f}, {ci[1]:.3f}]"
)

if not ci[0] <= 0 <= ci[1]:
    print("Reject the null hypothesis")
else:
    print("Reject the alternative hypothesis")

Point estimate of the difference in revenue: -19.999
0.95 confidence interval of the difference in revenue: [-20.142, -19.856]
Reject the null hypothesis


# Пункт (e)
**Сформулируйте вывод - стоит ли использовать эту акцию на постоянной основе или лучше от нее отказаться, и почему?**

Судя по полученным результатам, с высокой статистической значимостью можно делать вывод о том, что благодаря акции мы увеличили количество заказов, но при этом снизилась средняя выручка с пользователя, средняя выручка с заказа и общая выручка сервиса. Тогда лучше не использовать акцию на постоянной основе, так как она негативно влияет на прибыль.